In [0]:
# STARTER CODE - DO NOT EDIT THIS CELL

from pyspark.sql.functions import*
from pyspark.sql.types import *
from pyspark.sql import Window
from pyspark.sql.window import Window
from pyspark.sql import SparkSession

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")


In [0]:
# STARTER CODE - DO NOT EDIT THIS CELL

#from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

customSchema = StructType([
    StructField("lpep_pickup_datetime", StringType(), True),
    StructField("lpep_dropoff_datetime", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", FloatType(), True),
    StructField("fare_amount", FloatType(), True),
    StructField("payment_type", IntegerType(), True)
])

In [0]:
# STARTER CODE - YOU CAN LOAD ANY FILE WITH A SIMILAR SYNTAX.
# Correct file path for Databricks File System (update if you are using a different volume or file name)
file_path = "/Volumes/workspace/default/q2vol/nyc-tripdata.csv"

# Read the CSV file using Spark DataFrame
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

display(df.limit(5))


lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,payment_type
12/21/2018 15:17,12/21/2018 15:18,264,264,5,0.0,3.0,2
01/01/2019 0:10,01/01/2019 0:16,97,49,2,0.86,6.0,2
01/01/2019 0:27,01/01/2019 0:31,49,189,2,0.66,4.5,1
01/01/2019 0:46,01/01/2019 1:04,189,17,2,2.68,13.5,1
01/01/2019 0:19,01/01/2019 0:39,82,258,1,4.53,18.0,2


In [0]:
# LOAD THE "taxi_zone_lookup.csv" FILE SIMILARLY AS ABOVE. CAST ANY COLUMN TO APPROPRIATE DATA TYPE IF NECESSARY.

#ENTER THE CODE BELOW
file_path = "/Volumes/workspace/default/q2vol/taxi_zone_lookup.csv"

# Read the CSV file using Spark DataFrame
taxi_zone_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

display(taxi_zone_df.limit(5))

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone


In [0]:
#// STARTER CODE 
# // Some commands that you can use to see your dataframes and results of the operations. You can comment the df.show(5) and uncomment display(df) to see the data differently. You will find these two functions useful in reporting your results.
df.show(5)
#display(df.limit(5))

+--------------------+---------------------+------------+------------+---------------+-------------+-----------+------------+
|lpep_pickup_datetime|lpep_dropoff_datetime|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|payment_type|
+--------------------+---------------------+------------+------------+---------------+-------------+-----------+------------+
|    12/21/2018 15:17|     12/21/2018 15:18|         264|         264|              5|          0.0|        3.0|           2|
|     01/01/2019 0:10|      01/01/2019 0:16|          97|          49|              2|         0.86|        6.0|           2|
|     01/01/2019 0:27|      01/01/2019 0:31|          49|         189|              2|         0.66|        4.5|           1|
|     01/01/2019 0:46|      01/01/2019 1:04|         189|          17|              2|         2.68|       13.5|           1|
|     01/01/2019 0:19|      01/01/2019 0:39|          82|         258|              1|         4.53|       18.0|      

In [0]:
# // STARTER CODE - DO NOT EDIT THIS CELL
# Filter the data to only keep the rows where "PULocationID" and the "DOLocationID" are different and the "trip_distance" is strictly greater than 2.0 (>2.0).

# VERY VERY IMPORTANT: ALL THE SUBSEQUENT OPERATIONS MUST BE PERFORMED ON THIS FILTERED DATA

df_filter = df.filter((df.PULocationID != df.DOLocationID) & (df.trip_distance > 2.0))
df_filter.show(5)

+--------------------+---------------------+------------+------------+---------------+-------------+-----------+------------+
|lpep_pickup_datetime|lpep_dropoff_datetime|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|payment_type|
+--------------------+---------------------+------------+------------+---------------+-------------+-----------+------------+
|     01/01/2019 0:46|      01/01/2019 1:04|         189|          17|              2|         2.68|       13.5|           1|
|     01/01/2019 0:19|      01/01/2019 0:39|          82|         258|              1|         4.53|       18.0|           2|
|     01/01/2019 0:47|      01/01/2019 1:00|         255|          33|              1|         3.77|       13.5|           1|
|     01/01/2019 0:12|      01/01/2019 0:30|          76|         225|              1|          4.1|       16.0|           1|
|     01/01/2019 0:16|      01/01/2019 0:39|          25|          89|              1|         7.75|       25.5|      

In [0]:
# PART 1a: The top-5 most popular drop locations - "DOLocationID", sorted in descending order - if there is a tie, then one with lower "DOLocationID" gets listed first

# Output Schema: DOLocationID int, number_of_dropoffs int 

# Hint: Checkout the groupBy(), orderBy() and count() functions.

# ENTER THE CODE BELOW
popular_dropoffs = df_filter.groupBy("DOLocationID").agg(
    count("*").alias("number_of_dropoffs")
)

popular_dropoffs = popular_dropoffs.orderBy(
    desc("number_of_dropoffs"),
    asc("DOLocationID")  
)

popular_dropoffs.limit(5).show()


+------------+------------------+
|DOLocationID|number_of_dropoffs|
+------------+------------------+
|          61|              5937|
|         138|              5146|
|         239|              4133|
|         244|              4006|
|          42|              3859|
+------------+------------------+



In [0]:
# PART 1b: The top-5 most popular pickup locations - "PULocationID", sorted in descending order - if there is a tie, then one with lower "PULocationID" gets listed first 

# Output Schema: PULocationID int, number_of_pickups int

# ENTER THE CODE BELOW
popular_pickups = df_filter.groupBy("PULocationID").agg(
    count("*").alias("number_of_pickups")
)

popular_pickups = popular_pickups.orderBy(
    desc("number_of_pickups"),
    asc("PULocationID")  
)

popular_pickups.limit(5).show()


+------------+-----------------+
|PULocationID|number_of_pickups|
+------------+-----------------+
|          74|            17360|
|          75|            13299|
|         244|             9958|
|          41|             9645|
|          82|             9306|
+------------+-----------------+



In [0]:
# PART 2: List the top-3 locations with the maximum overall activity, i.e. sum of all pickups and all dropoffs at that LocationID. In case of a tie, the lower LocationID gets listed first.

# Output Schema: LocationID int, number_activities int

# Hint: In order to get the result, you may need to perform a join operation between the two dataframes that you created in earlier parts (to come up with the sum of the number of pickups and dropoffs on each location). 

# ENTER THE CODE BELOW

df_joined = popular_pickups.join(popular_dropoffs, on=popular_pickups['PULocationID'] == popular_dropoffs['DOLocationID'], how="left_outer")

df_joined = df_joined.fillna(0, subset=["number_of_pickups", "number_of_dropoffs"])

df_joined = df_joined.withColumn(
    "LocationID",
    coalesce(
        col("DOLocationID"),
        col("PULocationID")
    )
)

df_joined = df_joined.withColumn('number_activities', df_joined['number_of_pickups'] + df_joined['number_of_dropoffs'])

activity_df = df_joined.select(
    col("LocationID"),
    col("number_activities")
).orderBy(
    desc("number_activities"),
    asc("LocationID")  
)

activity_df.limit(3).show()

+----------+-----------------+
|LocationID|number_activities|
+----------+-----------------+
|        74|            20292|
|        75|            16326|
|       244|            13964|
+----------+-----------------+



In [0]:
# PART 3: List all the boroughs (including "Unknown" and "EWR") in the order of having the highest to lowest number of activities (i.e. sum of all pickups and all dropoffs at that LocationID), along with the total number of activity counts for each borough in NYC during that entire period of time.

# Output Schema: Borough string, total_number_activities int

# Hint: You can use the dataframe obtained from the previous part, and will need to do the join with the 'taxi_zone_lookup' dataframe. Also, checkout the "agg" function applied to a grouped dataframe.

# ENTER THE CODE BELOW
borough_df = activity_df.join(taxi_zone_df, on=activity_df['LocationID'] == taxi_zone_df['LocationID'], how="left_outer")

borough_df = borough_df.groupBy("Borough").agg(
    sum("number_activities").alias("total_number_activities")
)

borough_df = borough_df.select(
    col("Borough"),
    col("total_number_activities")
).orderBy(
    desc("total_number_activities")
)

borough_df.show()


+-------------+-----------------------+
|      Borough|total_number_activities|
+-------------+-----------------------+
|     Brooklyn|                 198506|
|    Manhattan|                 175953|
|       Queens|                 157633|
|        Bronx|                  67707|
|      Unknown|                   1215|
|Staten Island|                    888|
|          EWR|                    104|
+-------------+-----------------------+



In [0]:
# PART 4: List the top 2 days of week with the largest number of daily average pickups, along with the average number of pickups on each of the 2 days in descending order (no rounding off required). Here, the average pickup is calculated by taking an average of the number of pick-ups on different dates falling on the same day of the week. For example, 02/01/2021, 02/08/2021 and 02/15/2021 are all Mondays, so the average pick-ups for these is the sum of the pickups on each date divided by 3.

# Note: The day of week is a string of the day’s full spelling, e.g., "Monday" instead of the		number 1 or "Mon". Also, the pickup_datetime is in the format: yyyy-mm-dd.

# Output Schema: day_of_week string, avg_count float

# Hint: You may need to group by the "date" (without time stamp - time in the day) first. Checkout "date_format" and "to_date" functions.

# ENTER THE CODE BELOW
from pyspark.sql import functions as F

cast_df = df_filter.withColumn(
    'lpep_pickup_datetime',
    F.to_timestamp(
        F.col('lpep_pickup_datetime'),
        "MM/dd/yyyy H:mm" 
    )
)

daily_pickup_df = cast_df.withColumn(
    "pickup_date",
    date_trunc("day", col("lpep_pickup_datetime"))
).withColumn(
    "day_of_week",
    dayofweek(col("lpep_pickup_datetime"))
)

daily_pickup_df = daily_pickup_df.groupBy("pickup_date", "day_of_week").agg(
    count(col("PULocationID")).alias("daily_pickup_count")
)

daily_pickup_df = daily_pickup_df.groupBy("day_of_week").agg(
    avg(col("daily_pickup_count")).alias("avg_count")
)

daily_pickup_df = daily_pickup_df.orderBy(
    desc("avg_count")
)

daily_pickup_df = daily_pickup_df.withColumn(
    "day_of_week",
    when(col("day_of_week") == 1, "Sunday")
    .when(col("day_of_week") == 2, "Monday")
    .when(col("day_of_week") == 3, "Tuesday")
    .when(col("day_of_week") == 4, "Wednesday")
    .when(col("day_of_week") == 5, "Thursday")
    .when(col("day_of_week") == 6, "Friday")
    .when(col("day_of_week") == 7, "Saturday")
    .otherwise("Invalid")
).select(
    "day_of_week",
    "avg_count"
)

daily_pickup_df.limit(2).show()

+-----------+---------+
|day_of_week|avg_count|
+-----------+---------+
|  Wednesday|  10257.6|
|   Saturday|  9884.75|
+-----------+---------+



In [0]:
# PART 5: For each particular hour of a day (0 to 23, 0 being midnight) - in their order from 0 to 23, find the zone in Brooklyn borough with the LARGEST number of pickups. 

# Note: All dates for each hour should be included.

# Output Schema: hour_of_day int, zone string, max_count int

# Hint: You may need to use "Window" over hour of day, along with "group by" to find the MAXIMUM count of pickups

# ENTER THE CODE BELOW
from pyspark.sql import functions as F

cast_df = df_filter.withColumn(
    'lpep_pickup_datetime',
    F.to_timestamp(
        F.col('lpep_pickup_datetime'),
        "MM/dd/yyyy H:mm" 
    )
)

hourly_pickup_df = cast_df.join(
    taxi_zone_df,
    df_filter["PULocationID"] == taxi_zone_df["LocationID"],
    "inner"
).select(
    col("lpep_pickup_datetime"),
    col("Zone"),
    col("Borough")
)

hourly_pickup_df = hourly_pickup_df.filter(
    col("Borough") == "Brooklyn"
).withColumn(
    "hour_of_day",
    hour(col("lpep_pickup_datetime"))
)

hourly_pickup_df = hourly_pickup_df.groupBy("hour_of_day", "Zone").agg(
    count("*").alias("max_count")
)

window_spec = Window.partitionBy("hour_of_day").orderBy(
    desc("max_count"),
    asc("Zone")
)

hourly_pickup_df = hourly_pickup_df.withColumn(
    "rank",
    row_number().over(window_spec)
)

hourly_pickup_df = hourly_pickup_df.filter(
    col("rank") == 1
).select(
    col("hour_of_day"),
    col("Zone").alias("zone"),
    col("max_count")
)

hourly_pickup_df = hourly_pickup_df.orderBy(
    asc("hour_of_day")
)

hourly_pickup_df.show(24, truncate=False)

+-----------+---------------------------+---------+
|hour_of_day|zone                       |max_count|
+-----------+---------------------------+---------+
|0          |Williamsburg (North Side)  |569      |
|1          |Williamsburg (North Side)  |460      |
|2          |Williamsburg (North Side)  |429      |
|3          |Williamsburg (North Side)  |357      |
|4          |Williamsburg (North Side)  |228      |
|5          |East Williamsburg          |89       |
|6          |East New York              |149      |
|7          |Brooklyn Heights           |307      |
|8          |Brooklyn Heights           |511      |
|9          |Brooklyn Heights           |574      |
|10         |Brooklyn Heights           |502      |
|11         |Brooklyn Heights           |563      |
|12         |Brooklyn Heights           |491      |
|13         |Brooklyn Heights           |472      |
|14         |Fort Greene                |511      |
|15         |Fort Greene                |559      |
|16         

In [0]:
# PART 6 - Find which 3 different days in the month of January, in Manhattan, saw the largest positive percentage increase in pick-ups compared to the previous day, in the order from largest percentage increase to smallest percentage increase 

# Note: All years need to be aggregated to calculate the pickups for a specific day of January. The change from Dec 31 to Jan 1 can be excluded.

# Output Schema: day int, percent_change float

# Hint: You might need to use lag function, over a window ordered by day of month.

# ENTER THE CODE BELOW
from pyspark.sql import functions as F

cast_df = df_filter.withColumn(
    'lpep_pickup_datetime',
    F.to_timestamp(
        F.col('lpep_pickup_datetime'),
        "MM/dd/yyyy H:mm"  
    )
)

change_df = cast_df.join(
    taxi_zone_df,
    cast_df["PULocationID"] == taxi_zone_df["LocationID"],
    "inner"
).filter(
    (col("Borough") == "Manhattan") & (month(col("lpep_pickup_datetime")) == 1) 
)

change_df = change_df.groupBy(
    date_format(col("lpep_pickup_datetime"), "d").cast("int").alias("day_of_month") 
).agg(
    count("*").alias("current_pickups")
)

window_spec = Window.orderBy(col("day_of_month"))

change_df = change_df.withColumn(
    "previous_pickups",
    lag(col("current_pickups"), 1).over(window_spec)
)

change_df = change_df.withColumn(
    "percent_change",
    round(
        ((col("current_pickups") - col("previous_pickups")) / col("previous_pickups")) * 100,
        2
    )
)

change_df = change_df.filter(
    col("percent_change").isNotNull() & (col("percent_change") > 0)
).orderBy(
    desc("percent_change") 
)

change_df.select(
    col("day_of_month").alias("day"),
    col("percent_change")
).limit(3).show()



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1061: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---+--------------+
|day|percent_change|
+---+--------------+
| 22|         51.06|
|  2|         28.38|
| 28|         22.94|
+---+--------------+

